# 04. Backtest Analysis (Phase 3.5 수정 후)

Phase 3.5 버그 수정 + 감성분석 모델 교체 후 백테스트 재실행 및 KPI 분석.

**수정 사항**
1. LSTM TimeSeriesDataset seq_length 데이터 누수 수정 (#41)
2. 앙상블 out-of-fold stacking 구현 (#42)
3. RiskManager equity 계산에 포지션 크기 반영 (#43)
4. 감성분석 모델 교체: CardiffNLP → CryptoBERT + FinBERT (#47)
5. 백테스트 엔진 스케일링된 가격 사용 버그 수정 (#51)

**목표**
- XGBoost + LSTM + 앙상블 모델 학습 후 Test 세트 백테스트
- 1차 KPI 목표 충족 여부 확인 (Sharpe ≥ 1.0, MDD ≤ 25%, Win Rate ≥ 50%, PF ≥ 1.2)
- 모델별 기여도 분석
- 발견된 문제점 기록

In [ ]:
import sys
sys.path.insert(0, "..")

import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data.collector import load_from_parquet
from src.models.xgboost_model import XGBoostSignalModel
from src.models.lstm_model import LSTMSignalModel
from src.models.ensemble import EnsembleModel
from src.backtest.engine import BacktestEngine

pd.set_option("display.float_format", "{:.4f}".format)

## 1. 데이터 및 모델 로드

In [ ]:
# 데이터 로드
train_df = load_from_parquet("../data/processed/train.parquet")
val_df = load_from_parquet("../data/processed/val.parquet")
test_df = load_from_parquet("../data/processed/test.parquet")

print(f"Train: {len(train_df):,}행 ({train_df.index.min()} ~ {train_df.index.max()})")
print(f"Val:   {len(val_df):,}행 ({val_df.index.min()} ~ {val_df.index.max()})")
print(f"Test:  {len(test_df):,}행 ({test_df.index.min()} ~ {test_df.index.max()})")
print(f"피처 수: {len(train_df.columns) - 1} (target 제외)")

# 모델 로드
xgb = XGBoostSignalModel.load("../data/models")
lstm = LSTMSignalModel.load("../data/models")
ensemble = EnsembleModel.load("../data/models")

## 2. 예측 생성 및 정렬

In [ ]:
# XGBoost 예측
xgb_test_proba = xgb.predict_proba(test_df)

# LSTM 예측 (seq_length offset)
seq_length = lstm.config.get("seq_length", 60)
lstm_test_proba = lstm.predict_proba(test_df)
n_lstm = len(lstm_test_proba)

# LSTM은 seq_length만큼 짧으므로 정렬
xgb_test_aligned = xgb_test_proba[seq_length - 1 : seq_length - 1 + n_lstm]
test_df_aligned = test_df.iloc[seq_length - 1 : seq_length - 1 + n_lstm]

print(f"seq_length: {seq_length}")
print(f"정렬 후 Test 크기: {len(test_df_aligned):,}행")
print(f"XGBoost proba shape: {xgb_test_aligned.shape}")
print(f"LSTM proba shape: {lstm_test_proba.shape}")

# 앙상블 신호 생성
base_predictions = {"xgboost": xgb_test_aligned, "lstm": lstm_test_proba}
signals = ensemble.predict(base_predictions)

# 신호 분포
unique, counts = np.unique(signals, return_counts=True)
label_map = {-1: "Sell", 0: "Hold", 1: "Buy"}
print("\n신호 분포:")
for u, c in zip(unique, counts):
    print(f"  {label_map.get(u, u)}: {c:,} ({c/len(signals)*100:.1f}%)")

## 3. 백테스트 실행

In [ ]:
engine = BacktestEngine(config_path="../configs/trading_config.yaml")
result = engine.run(test_df_aligned, signals)

# KPI 출력
print("=" * 60)
print("         BACKTEST RESULTS (Phase 3.5 수정 후)")
print("=" * 60)
for k, v in result.metrics.items():
    if isinstance(v, float):
        print(f"  {k:.<30} {v:>12.4f}")
    else:
        print(f"  {k:.<30} {v:>12}")
print("=" * 60)

# 1차 KPI 목표 비교
targets = {
    "sharpe_ratio": (">=", 1.0),
    "max_drawdown": (">=", -0.25),
    "win_rate": (">=", 0.50),
    "profit_factor": (">=", 1.2),
}
print("\n1차 KPI 목표 비교:")
for metric, (op, target) in targets.items():
    actual = result.metrics.get(metric, 0)
    passed = actual >= target
    status = "PASS ✓" if passed else "FAIL ✗"
    print(f"  {metric:.<25} {actual:>8.4f} {op} {target:>6.2f}  → {status}")

## 4. Equity Curve 시각화

In [ ]:
# Equity curve vs Buy & Hold (원본 가격 사용)
initial = result.config["initial_capital"]
price_col = "close_raw" if "close_raw" in test_df_aligned.columns else "close"
bh_equity = initial * (test_df_aligned[price_col] / test_df_aligned[price_col].iloc[0])

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05,
    row_heights=[0.7, 0.3],
    subplot_titles=("Equity Curve", "Drawdown")
)

fig.add_trace(
    go.Scatter(x=result.equity_curve.index, y=result.equity_curve.values,
               name="Strategy", line=dict(color="blue", width=1.5)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=bh_equity.index, y=bh_equity.values,
               name="Buy & Hold", line=dict(color="gray", width=1, dash="dash")),
    row=1, col=1
)

# Drawdown
peak = result.equity_curve.cummax()
drawdown = (result.equity_curve - peak) / peak
fig.add_trace(
    go.Scatter(x=drawdown.index, y=drawdown.values,
               name="Drawdown", fill="tozeroy",
               line=dict(color="red", width=1), fillcolor="rgba(255,0,0,0.2)"),
    row=2, col=1
)

fig.update_layout(height=600, showlegend=True)
fig.update_yaxes(title_text="USD", row=1, col=1)
fig.update_yaxes(title_text="Drawdown %", row=2, col=1)
fig.show()

## 5. 거래 분석

In [ ]:
# 거래별 PnL 분포
trades = result.trades
print(f"총 거래 수: {len(trades)}")
print(f"평균 보유 시간: {trades['duration'].mean()}")
print()

# PnL 히스토그램
fig = make_subplots(rows=1, cols=2, subplot_titles=("거래별 PnL (%)", "거래별 PnL (USD)"))

fig.add_trace(
    go.Histogram(x=trades["pnl"] * 100, nbinsx=30, marker_color="steelblue", name="PnL %"),
    row=1, col=1
)
fig.add_trace(
    go.Histogram(x=trades["pnl_abs"], nbinsx=30, marker_color="coral", name="PnL USD"),
    row=1, col=2
)

fig.update_xaxes(title_text="PnL (%)", row=1, col=1)
fig.update_xaxes(title_text="PnL (USD)", row=1, col=2)
fig.update_layout(height=350, showlegend=False)
fig.show()

# 극단 거래 확인
print("Top 5 최악의 거래:")
worst = trades.nsmallest(5, "pnl")[["entry_time", "exit_time", "entry_price", "exit_price", "pnl", "pnl_abs", "duration"]]
print(worst.to_string())
print()
print("Top 5 최고의 거래:")
best = trades.nlargest(5, "pnl")[["entry_time", "exit_time", "entry_price", "exit_price", "pnl", "pnl_abs", "duration"]]
print(best.to_string())

## 6. 앙상블 모델 기여도 분석

In [ ]:
# 앙상블 메타 모델 정보
meta = ensemble.get_metadata()
print("앙상블 메타데이터:")
for k, v in meta.items():
    print(f"  {k}: {v}")

# 모델별 기여도
contributions = ensemble.get_contributions()
print(f"\n모델별 기여도: {contributions}")

fig = go.Figure(go.Bar(
    x=list(contributions.keys()),
    y=list(contributions.values()),
    marker_color=["steelblue", "coral"],
    text=[f"{v:.1%}" for v in contributions.values()],
    textposition="auto"
))
fig.update_layout(
    title="앙상블 모델별 기여도",
    yaxis_title="기여도", height=350
)
fig.show()

## 7. 발견된 문제점 및 다음 단계

### Phase 3.5 버그 수정 효과
| 지표 | 수정 전 (스케일링된 가격) | 수정 후 (원본 가격) | 비고 |
|------|--------------------------|---------------------|------|
| MDD | -158% (비현실적) | -43.62% | 물리적으로 가능한 범위 |
| Final Capital | -$989 (음수) | $6,270 | 합리적 결과 |
| worst_trade | -113% (불가능) | -39.10% | 정상 범위 |
| 총 거래 수 | 다수 | 1건 | Sell 신호 0건 |

### 핵심 문제점
1. **Sell 신호 0건**: 앙상블이 Hold 95.7% + Buy 4.3%만 생성, Sell은 전혀 없음
   - 매수 후 매도 없이 마지막 bar에서 강제 청산 → 거래 1건
   - 원인: 메타 모델이 Sell 클래스를 학습하지 못함
2. **LSTM 기여도 1.88%**: 사실상 XGBoost 단독 모델
   - LSTM val_loss가 0.927로 수렴 (랜덤 수준 ~1.099 대비 개선은 있으나 미미)
3. **모든 KPI 미달**: Sharpe -2.33, MDD -43.62%, Win Rate 0%, PF 0.0
4. **긍정적 신호**: excess_return +1.32% (Buy & Hold 대비 소폭 우위)

### 원인 분석
- 3-class 분류(Buy/Hold/Sell)에서 Hold가 압도적 다수 → 모델이 Hold 편향
- target_threshold 0.5%가 너무 낮아 노이즈에 민감할 가능성
- 1시간 캔들 단일 타임프레임으로는 추세 파악 한계

### 다음 단계 (Phase 6: 전략 고도화)
- [ ] 마켓 레짐 감지 모듈 (상승장/하락장/횡보장 분류)
- [ ] 멀티 타임프레임 분석 (4h, 1d 추세 확인 → 1h 진입)
- [ ] target_threshold 튜닝 및 2-class(Buy/Sell) vs 3-class 비교
- [ ] 동적 포지션 사이징 (Kelly Criterion / 변동성 기반)
- [ ] LSTM 아키텍처 개선 (Attention, Transformer 등)